In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview ---")
print(pages[0].page_content[:500])

Loaded 9 pages from the PDF.

--- First page preview ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [4]:
print(pages[1].page_content[:500])

Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.
3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, an


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators = ["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(pages)

print(len(chunks))


24


In [9]:
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [10]:
chunks[9].page_content

"Telecom Technical Reference Guide  - Internal Use Only\n4. International Roaming  - How It Works\nWhen a customer travels outside the home network's coverage area, their device connects to a partner network in\nthe visited country. This is called roaming. Understanding the roaming architecture helps agents diagnose\nroaming failures and billing issues.\nTechnical Mechanism: The visited network authenticates the customer via an inter-operator signalling protocol\n(SS7 or Diameter). The home network validates the subscription and authorises service. All data, voice, and SMS\ntraffic is tunnelled back to the home network for billing. This tunnelling adds latency compared to local network\nusage.\nRoaming Zones: Our network divides the world into roaming zones based on inter-operator agreements and cost\nstructures. Zone A (EU, UK, Australia, New Zealand) has the lowest roaming rates. Zone B (USA, Canada,"

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 17882.09it/s]


Vector store ready. 24 vectors stored.


In [13]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

test_query = "How does international roaming work?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"---Chunk {i} ---")
    print(doc.page_content[:300])
    print()

---Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
4. International Roaming  - How It Works
When a customer travels outside the home network's coverage area, their device connects to a partner network in
the visited country. This is called roaming. Understanding the roaming architecture helps ag

---Chunk 2 ---
structures. Zone A (EU, UK, Australia, New Zealand) has the lowest roaming rates. Zone B (USA, Canada,
Japan, Singapore) has moderate rates. Zone C (Rest of World) has the highest per-MB and per-minute charges.
Customers should always purchase a roaming bundle before travelling to Zone B or C countr

---Chunk 3 ---
selected. A daily bundle covers midnight to midnight in the home timezone. Customers who activate a bundle
after using some data on the day will still be charged the bundle price, but any pre-bundle usage at standard rates
is not reversed.
Common Roaming Failures: The three most common reasons a cus



In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

def format_docs(docs):
    return( "\n\n---\n\n".join(doc.page_content for doc in docs) )

SYSTEM_PROMPT  = """\
You are a helpful telecom assistant.
Answer the questions using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])


llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format = "parsed",
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled")

RAG chain assembled


In [15]:
question = "How does international roaming work and what charges do we expect"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges do we expect

A: International roaming works by connecting your device to a partner network in the visited country. The visited network authenticates your subscription via inter-operator protocols (SS7/Diameter), and all traffic (data, voice, SMS) is tunneled back to your home network for billing. This process adds latency compared to local network usage.

**Charges depend on roaming zones:**  
- **Zone A** (EU, UK, Australia, New Zealand): Lowest roaming rates.  
- **Zone B** (USA, Canada, Japan, Singapore): Moderate rates.  
- **Zone C** (Rest of World): Highest per-MB and per-minute charges.  

To avoid high charges, customers should **purchase a roaming bundle** for the destination zone before travel. Bundles are valid for selected calendar days and cover usage within the specified zone. Without a bundle, charges are applied at standard zone rates, which can lead to "bill shock."  

**Key notes:**  
- Roaming must be enabled v